# ❄️ Unidad 2 — Q-Learning desde Cero con NumPy

**Curso: Aprendizaje por Refuerzo Profundo en Español**  
Adaptación pedagógica basada en el curso de HuggingFace 🤗  
Autor: Jose Miguel Lara Rangel · [aicraft.mx](https://aicraft.mx)

---

### ¿Qué construiremos?

En este notebook implementaremos el algoritmo **Q-Learning** línea por línea usando solo NumPy — sin librerías mágicas.  
Después aplicaremos el mismo código a dos entornos distintos:

| Entorno | Estados | Acciones | Tarea |
|---------|---------|----------|-------|
| ❄️ **FrozenLake-v1** | 16 | 4 | Cruzar un lago helado sin caer |
| 🚖 **Taxi-v3** | 500 | 6 | Recoger y dejar pasajeros en una ciudad |

Al terminar habrás:
- ✅ Implementado la Q-table, la política greedy y la política ε-greedy
- ✅ Construido el bucle de entrenamiento Q-Learning completo
- ✅ Entendido por qué el mismo código funciona con 16 o 500 estados
- ✅ Evaluado el agente con métricas estadísticas robustas

> 📎 **Slides de referencia:** Capítulos 2.1–2.6 del curso.  
> ⏱️ **Tiempo estimado:** ~25 minutos (la mayor parte es código, no entrenamiento)

---
## 1 · Configurar el entorno de trabajo

### 1.1 · Instalar dependencias

Este notebook necesita menos librerías que el de LunarLander porque implementamos  
Q-Learning desde cero con NumPy — no usamos Stable-Baselines3.

| Librería | ¿Para qué? |
|----------|-----------|
| `gymnasium` | Los entornos FrozenLake y Taxi |
| `imageio` | Grabar videos del agente jugando |
| `tqdm` | Barra de progreso durante el entrenamiento |
| `pickle5` | Guardar y cargar la Q-table en disco |

In [ ]:
!pip install -q gymnasium imageio tqdm pickle5

print("✅ Dependencias instaladas")

### 1.2 · Herramientas de video (para visualizar el agente)

> ⚠️ Después de esta celda, **acepta reiniciar el entorno** si Colab lo solicita  
> y luego ejecuta desde la sección 1.3.

In [ ]:
!sudo apt-get update -q
!sudo apt-get install -q -y python3-opengl
!apt install -q ffmpeg xvfb
!pip install -q pyvirtualdisplay

print("✅ Herramientas de video instaladas")

### 1.3 · Iniciar pantalla virtual

In [ ]:
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()
print("✅ Pantalla virtual iniciada")

### 1.4 · Importar librerías

In [ ]:
import numpy as np            # operaciones sobre la Q-table
import gymnasium as gym        # entornos FrozenLake y Taxi
import random                  # números aleatorios para ε-greedy
import imageio                 # grabar video del agente
import pickle5 as pickle       # guardar/cargar la Q-table
from tqdm.notebook import tqdm # barra de progreso

print("✅ Librerías importadas")

---
## 2 · Repaso rápido: ¿qué es una Q-table?

Antes de escribir código, recordemos la idea central del Módulo 2.

### La Q-table como "hoja de trucos" del agente

Una Q-table es simplemente una **matriz de números** donde:
- Cada **fila** es un estado posible del entorno
- Cada **columna** es una acción posible
- Cada **celda** contiene un número: cuánto vale tomar esa acción en ese estado

```
           ⬅️ Izq.   ⬇️ Abajo   ➡️ Der.   ⬆️ Arriba
Estado 0  [  0.0      0.0       0.73     0.0   ]  ← mejor ir a la derecha
Estado 1  [  0.0      0.62      0.0      0.0   ]  ← mejor ir abajo
Estado 2  [  0.0      0.0       0.0      0.78  ]  ← mejor ir arriba
...
```

### ¿Cómo se actualiza?

En cada paso, Q-Learning corrige el valor de la acción tomada usando la ecuación de Bellman:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \Big[R_{t+1} + \gamma \cdot \max_{a'} Q(s',a') - Q(s,a)\Big]$$

En palabras: **"el nuevo valor Q = el viejo valor + una pequeña corrección hacia lo que debería ser".**

Al inicio todos los valores son 0. Con cada episodio, los valores correctos van subiendo  
y los incorrectos se quedan en 0 o bajan — hasta que la tabla refleja la política óptima.

> 📎 Si quieres repasar la teoría: slides de los capítulos 2.1 y 2.2.

---
## 3 · Parte 1 — FrozenLake ❄️ (versión no resbaladiza)

### 3.1 · Crear y explorar el entorno

FrozenLake es una cuadrícula 4×4. El agente (un duende 🧝) debe llegar desde  
la casilla de inicio **S** hasta la meta **G** sin caer en los hoyos **H**.

```
S F F F     S = inicio (Start)
F H F H     F = hielo (Frozen) — seguro
F F F H     H = hoyo  (Hole)   — episodio terminado
H F F G     G = meta  (Goal)   — recompensa +1
```

Usamos `is_slippery=False` para empezar: el agente siempre se mueve  
exactamente hacia donde elige. Más adelante veremos qué pasa cuando  
el hielo resbala.

---
### 🏋️ EJERCICIO (opcional)

Crea el entorno con los parámetros correctos usando `gym.make()`.  

**Pistas:**
- El nombre del entorno es `'FrozenLake-v1'`
- Parámetro para el tamaño del mapa: `map_name='4x4'`
- Parámetro para desactivar el resbalón: `is_slippery=False`
- Para poder grabar video: `render_mode='rgb_array'`

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — escribe tu código aquí
# env = gym.make(...)

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

print("✅ Entorno FrozenLake-v1 creado (versión no resbaladiza)")

### 3.2 · Explorar el espacio de estados y acciones

A diferencia de LunarLander (donde el estado eran 8 números continuos),  
en FrozenLake el estado es un **número entero del 0 al 15** — el número de casilla.

```
 0  1  2  3
 4  5  6  7
 8  9 10 11
12 13 14 15
```

Las 4 acciones disponibles:

| Acción | Código | Dirección |
|--------|--------|-----------|
| Izquierda | `0` | ← |
| Abajo | `1` | ↓ |
| Derecha | `2` | → |
| Arriba | `3` | ↑ |

In [ ]:
print("═══ ESPACIO DE OBSERVACIONES ═══")
print(f'Tipo: {env.observation_space}')
print(f'Número de estados posibles: {env.observation_space.n}')
print(f'Ejemplo de estado inicial: {env.observation_space.sample()}')
print()
print("═══ ESPACIO DE ACCIONES ═══")
print(f'Número de acciones posibles: {env.action_space.n}')
print(f'Ejemplo de acción aleatoria: {env.action_space.sample()}')
print()
# Ver qué devuelve dar un paso
obs, info = env.reset()
nueva_obs, recompensa, terminado, truncado, info = env.step(2)  # ir a la derecha
print(f"Estado inicial: {obs} → acción: 2 (derecha) → nuevo estado: {nueva_obs}")
print(f"Recompensa: {recompensa} | ¿Terminado?: {terminado}")

### 3.3 · Inicializar la Q-table

El primer paso del algoritmo Q-Learning es crear la Q-table e inicializarla en ceros.  
Para FrozenLake necesitamos una tabla de **16 estados × 4 acciones = 64 celdas**.

Usaremos `np.zeros((filas, columnas))` para crear la matriz.

---
### 🏋️ EJERCICIO (opcional)

Implementa la función `initialize_q_table(state_space, action_space)` que devuelve  
una matriz de ceros con las dimensiones correctas.

**Pista:** `np.zeros` recibe una tupla `(filas, columnas)`.

Si prefieres no hacerlo como ejercicio, salta a la celda de Solución.

In [ ]:
# 🏋️ EJERCICIO — completa la función
def initialize_q_table(state_space, action_space):
    Qtable = # tu código aquí
    return Qtable

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
def initialize_q_table(state_space, action_space):
    # np.zeros crea una matriz llena de 0.0 con las dimensiones dadas
    Qtable = np.zeros((state_space, action_space))
    return Qtable

# Obtener dimensiones del entorno
state_space  = env.observation_space.n   # 16
action_space = env.action_space.n        # 4

# Crear la Q-table
Qtable_frozenlake = initialize_q_table(state_space, action_space)

print(f"Dimensiones de la Q-table: {Qtable_frozenlake.shape}  ({state_space} estados × {action_space} acciones)")
print(f"Total de celdas: {Qtable_frozenlake.size}")
print()
print("Q-table inicial (todos en cero):")
print(Qtable_frozenlake)

### 3.4 · Política greedy — explotar lo aprendido

La **política greedy** es la que usamos para *evaluar* el agente:  
siempre elige la acción con el **mayor valor Q** en el estado actual.

$$\pi^*(s) = \arg\max_a Q(s,a)$$

`np.argmax(array)` devuelve el **índice** del valor más alto del array —  
que en este caso es el número de acción con mayor Q.

---
### 🏋️ EJERCICIO (opcional)

Implementa `greedy_policy(Qtable, state)` que devuelve la acción con mayor Q-value  
para el estado dado.

**Pista:** `Qtable[state]` devuelve la fila de ese estado — un array de 4 valores.
Luego `np.argmax(...)` sobre ese array te da el índice del máximo.

In [ ]:
# 🏋️ EJERCICIO — completa la función
def greedy_policy(Qtable, state):
    action = # tu código aquí
    return action

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
def greedy_policy(Qtable, state):
    # np.argmax devuelve el índice del valor máximo
    # Qtable[state] es la fila del estado actual — 4 valores Q
    action = np.argmax(Qtable[state][:])
    return action

# Prueba rápida con una Q-table de ejemplo
q_ejemplo = np.array([[0.0, 0.5, 2.3, 0.1],   # estado 0: mejor acción es 2 (→)
                       [1.8, 0.0, 0.4, 0.2]])   # estado 1: mejor acción es 0 (←)
print(f"Estado 0 → acción greedy: {greedy_policy(q_ejemplo, 0)} (esperado: 2)")
print(f"Estado 1 → acción greedy: {greedy_policy(q_ejemplo, 1)} (esperado: 0)")

### 3.5 · Política ε-greedy — explorar mientras aprendemos

La política greedy tiene un problema: si siempre elige la acción con mayor Q-value,  
nunca explorará acciones nuevas. Al inicio, todos los Q-values son 0 — ¡no hay nada que explotar!

La **política ε-greedy** resuelve esto con una regla simple:

```
genera un número aleatorio entre 0 y 1
  si número > ε  →  EXPLOTAR: elige la acción con mayor Q (greedy)
  si número ≤ ε  →  EXPLORAR: elige una acción aleatoria
```

**ε (epsilon)** empieza alto (1.0 = explorar siempre) y baja gradualmente  
conforme el agente aprende, hasta un mínimo de 0.05.

---
### 🏋️ EJERCICIO (opcional)

Implementa `epsilon_greedy_policy(Qtable, state, epsilon)`.

**Pistas:**
- `random.uniform(0, 1)` genera un número aleatorio entre 0 y 1
- Si el número es **mayor** que epsilon → llama a `greedy_policy()`
- Si el número es **menor o igual** que epsilon → `env.action_space.sample()`

In [ ]:
# 🏋️ EJERCICIO — completa la función
def epsilon_greedy_policy(Qtable, state, epsilon):
    random_num = # genera número aleatorio
    if random_num > epsilon:
        action = # explotar
    else:
        action = # explorar
    return action

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
def epsilon_greedy_policy(Qtable, state, epsilon):
    random_num = random.uniform(0, 1)
    if random_num > epsilon:
        # Explotar: elegir la mejor acción conocida
        action = greedy_policy(Qtable, state)
    else:
        # Explorar: elegir una acción al azar
        action = env.action_space.sample()
    return action

# Prueba: con epsilon=1.0 siempre debería explorar (valores casi siempre distintos)
acciones = [epsilon_greedy_policy(Qtable_frozenlake, 0, epsilon=1.0) for _ in range(10)]
print(f"Con ε=1.0 (solo explorar):  {acciones}")
# Con epsilon=0.0 siempre debería explotar (siempre la misma acción greedy)
acciones = [epsilon_greedy_policy(Qtable_frozenlake, 0, epsilon=0.0) for _ in range(10)]
print(f"Con ε=0.0 (solo explotar):  {acciones}  ← todos iguales (Q-table = 0, argmax = 0)")

### 3.6 · Definir los hiperparámetros

Antes de entrenar, configuramos todos los parámetros del entrenamiento:

| Parámetro | Valor | ¿Qué controla? | ¿Qué pasa si lo cambias? |
|-----------|-------|----------------|--------------------------|
| `n_training_episodes` | 10,000 | Total de episodios | Más → mejor (hasta cierto punto) |
| `learning_rate` (α) | 0.7 | Velocidad de actualización Q | Muy alto → inestable. Muy bajo → lento |
| `max_steps` | 99 | Pasos máximos por episodio | Limita el tiempo por intento |
| `gamma` (γ) | 0.95 | Descuento del futuro | Cerca de 1 → valora el futuro igual que el presente |
| `max_epsilon` | 1.0 | ε inicial | Siempre 1.0 — explorar completamente al inicio |
| `min_epsilon` | 0.05 | ε mínimo | Siempre habrá al menos 5% de exploración |
| `decay_rate` | 0.0005 | Velocidad de decaimiento de ε | Más alto → ε baja más rápido |

In [ ]:
# ── Hiperparámetros de entrenamiento ────────────────────
n_training_episodes = 10_000  # episodios totales
learning_rate       = 0.7     # α — tasa de aprendizaje

# ── Hiperparámetros del entorno ──────────────────────────
env_id    = "FrozenLake-v1"
max_steps = 99
gamma     = 0.95
n_eval_episodes = 100
eval_seed = []               # sin seed fijo (FrozenLake es determinista)

# ── Hiperparámetros de exploración ε-greedy ─────────────
max_epsilon = 1.0            # ε al inicio: explorar siempre
min_epsilon = 0.05           # ε mínimo: siempre 5% de exploración
decay_rate  = 0.0005         # velocidad de decaimiento

# ── Visualizar cómo decae ε durante el entrenamiento ────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

episodios = np.arange(n_training_episodes)
epsilons = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episodios)

plt.figure(figsize=(8, 3))
plt.plot(episodios, epsilons, color='#22d3ee')
plt.xlabel("Episodio")
plt.ylabel("ε (epsilon)")
plt.title("Decaimiento de ε durante el entrenamiento")
plt.axhline(y=min_epsilon, color="gray", linestyle="--", alpha=0.5, label=f"ε mínimo = {min_epsilon}")
plt.legend()
plt.tight_layout()
plt.savefig("epsilon_decay.png", dpi=100)
plt.close()

from IPython.display import Image
Image("epsilon_decay.png")

### 3.7 · El bucle de entrenamiento — `train()`

Esta es la función más importante del notebook. Implementa el algoritmo Q-Learning completo.

**El pseudocódigo que implementaremos:**
```
Para cada episodio:
    1. Calcular ε para este episodio (va bajando)
    2. Reiniciar el entorno → obtener estado inicial
    3. Para cada paso del episodio:
        a. Elegir acción con ε-greedy
        b. Ejecutar acción → obtener recompensa y nuevo estado
        c. Actualizar Q-table:  ← ¡LA LÍNEA MÁS IMPORTANTE!
           Q[s,a] = Q[s,a] + α × (R + γ × max Q[s'] - Q[s,a])
        d. Si terminó → romper el bucle
        e. El nuevo estado es ahora el estado actual
```

---
### 🏋️ EJERCICIO (opcional)

Completa los tres huecos de la función `train()`. Los más importantes:
1. Elegir la acción con `epsilon_greedy_policy()`
2. Ejecutar la acción con `env.step()`
3. Actualizar la Q-table con la ecuación de Bellman

La ecuación de actualización:
$$Q[s,a] = Q[s,a] + \alpha \times (R + \gamma \times \max_a Q[s'] - Q[s,a])$$

In [ ]:
# 🏋️ EJERCICIO — completa los tres huecos marcados con ###
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        # 1. Calcular epsilon para este episodio
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        terminated = False
        truncated  = False

        for step in range(max_steps):
            # 2. Elegir acción con ε-greedy
            action = ### tu código aquí

            # 3. Ejecutar la acción
            new_state, reward, terminated, truncated, info = ### tu código aquí

            # 4. Actualizar Q-table (ecuación de Bellman)
            Qtable[state][action] = ### tu código aquí

            if terminated or truncated:
                break
            state = new_state
    return Qtable

---
### ✅ Solución

In [ ]:
# ✅ SOLUCIÓN
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        # 1. ε decae exponencialmente con cada episodio
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        terminated = False
        truncated  = False

        for step in range(max_steps):
            # 2. Elegir acción con la política ε-greedy
            action = epsilon_greedy_policy(Qtable, state, epsilon)

            # 3. Ejecutar la acción en el entorno
            new_state, reward, terminated, truncated, info = env.step(action)

            # 4. ★ Actualizar Q-table — la ecuación de Bellman
            #    Q[s,a] = Q[s,a] + α × (R + γ × max_a' Q[s',a'] - Q[s,a])
            Qtable[state][action] = (Qtable[state][action]
                + learning_rate * (reward
                                   + gamma * np.max(Qtable[new_state])
                                   - Qtable[state][action]))

            if terminated or truncated:
                break
            state = new_state
    return Qtable

### 3.8 · Entrenar el agente

Con todas las piezas listas, lanzamos el entrenamiento.  
10,000 episodios con la barra de progreso de `tqdm`.  
Debería tardar **menos de 1 minuto** — FrozenLake es muy ligero.

In [ ]:
print("Entrenando agente en FrozenLake-v1...")
Qtable_frozenlake = train(
    n_training_episodes,
    min_epsilon,
    max_epsilon,
    decay_rate,
    env,
    max_steps,
    Qtable_frozenlake
)
print("✅ Entrenamiento completado")

### 3.9 · Visualizar la Q-table entrenada

Ahora miramos los valores que aprendió el agente.  
Para cada estado mostramos qué acción tiene el mayor Q-value — esa es la política óptima.

In [ ]:
# Ver la Q-table cruda
print("Q-table entrenada (16 estados × 4 acciones):")
print("Columnas: [← Izq.  ↓ Abajo  → Der.  ↑ Arriba]")
print(np.round(Qtable_frozenlake, 3))
print()

# Mostrar la política óptima en el mapa 4×4
acciones_str = ['←', '↓', '→', '↑']
mapa = ['S', 'F', 'F', 'F',
        'F', 'H', 'F', 'H',
        'F', 'F', 'F', 'H',
        'H', 'F', 'F', 'G']

print("Política óptima aprendida (flecha = acción con mayor Q-value):")
print("─" * 21)
for i, (celda, q_fila) in enumerate(zip(mapa, Qtable_frozenlake)):
    mejor_accion = acciones_str[np.argmax(q_fila)]
    # Mostrar símbolo del mapa si es hoyo/meta, flecha si es casilla libre
    if celda in ['H', 'G']:
        simbolo = f" {celda} "
    else:
        simbolo = f" {mejor_accion} "
    print(simbolo, end="")
    if (i + 1) % 4 == 0: print("|")
print("─" * 21)
print("S=inicio  G=meta  H=hoyo")

### 3.10 · Evaluar el agente

Definimos la función de evaluación y medimos el desempeño real del agente.  
La evaluación usa `greedy_policy` (sin exploración) y un `seed` opcional  
para reproducir los mismos episodios entre distintas ejecuciones.

| Tasa de éxito | Interpretación |
|--------------|----------------|
| ≥ 1.00 | ✅ El agente siempre llega a la meta (versión no resbaladiza) |
| 0.80–0.99 | Buen agente, ocasionalmente elige un camino subóptimo |
| < 0.80 | El agente necesita más entrenamiento o ajuste de parámetros |

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    """
    Evalúa el agente durante n_eval_episodes episodios.
    Devuelve la recompensa media y su desviación estándar.
    """
    episode_rewards = []

    for episode in tqdm(range(n_eval_episodes)):
        if seed:
            state, info = env.reset(seed=seed[episode])
        else:
            state, info = env.reset()
        total_rewards_ep = 0
        terminated = False
        truncated  = False

        for step in range(max_steps):
            # Usar política greedy (sin exploración) para evaluar
            action = greedy_policy(Q, state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward
            if terminated or truncated:
                break
            state = new_state

        episode_rewards.append(total_rewards_ep)

    mean_reward = np.mean(episode_rewards)
    std_reward  = np.std(episode_rewards)
    return mean_reward, std_reward

In [ ]:
print("Evaluando agente en 100 episodios...")
mean_reward, std_reward = evaluate_agent(
    env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed
)
print(f"\nRecompensa media: {mean_reward:.2f} ± {std_reward:.2f}")

if mean_reward >= 1.0:
    print("✅ ¡Meta alcanzada! El agente cruza el lago perfectamente.")
elif mean_reward >= 0.8:
    print("🟡 Buen resultado. Prueba entrenar más episodios.")
else:
    print("🔴 El agente necesita más entrenamiento.")

### 3.11 · Ver al agente jugar en video

Grabamos un episodio completo del agente usando la política greedy.

In [ ]:
def grabar_video_qtable(env_id, Qtable, fps=2, **env_kwargs):
    """Graba un episodio del agente guiado por la Q-table."""
    env_video = gym.make(env_id, render_mode='rgb_array', **env_kwargs)
    frames = []
    state, _ = env_video.reset()
    frames.append(env_video.render())
    terminated = truncated = False

    while not (terminated or truncated):
        action = greedy_policy(Qtable, state)
        state, _, terminated, truncated, _ = env_video.step(action)
        frames.append(env_video.render())

    env_video.close()
    ruta = "replay_frozenlake.mp4"
    imageio.mimsave(ruta, [np.array(f) for f in frames], fps=fps)
    return ruta

print("Grabando video...")
video = grabar_video_qtable(
    "FrozenLake-v1", Qtable_frozenlake,
    map_name='4x4', is_slippery=False
)

from IPython.display import Video
Video(video, embed=True, width=400)

---
## 4 · Parte 2 — Taxi-v3 🚖 (el mismo código, más estados)

### El salto de FrozenLake a Taxi

La transición más importante de este notebook: vamos a usar **exactamente las mismas funciones**  
(`initialize_q_table`, `greedy_policy`, `epsilon_greedy_policy`, `train`, `evaluate_agent`)  
con un entorno 30 veces más grande.

| Característica | ❄️ FrozenLake | 🚖 Taxi |
|----------------|--------------|---------|
| Número de estados | 16 | **500** |
| Número de acciones | 4 | **6** |
| Tamaño Q-table | 64 celdas | **3,000 celdas** |
| Episodios de entrenamiento | 10,000 | **25,000** |
| Funciones que cambian | — | **ninguna** |

Esto demuestra que **Q-Learning es un algoritmo general**: no está amarrado a un entorno específico.

### ¿Cómo se codifican los 500 estados de Taxi?

El estado de Taxi combina 3 factores:

```
25 posiciones del taxi (cuadrícula 5×5)
×  5 posiciones del pasajero (R, G, Y, B o dentro del taxi)
×  4 destinos posibles (R, G, Y, B)
= 500 estados totales
```

Las 6 acciones:

| Código | Acción |
|--------|--------|
| 0 | Mover al Sur ↓ |
| 1 | Mover al Norte ↑ |
| 2 | Mover al Este → |
| 3 | Mover al Oeste ← |
| 4 | Recoger pasajero 📥 |
| 5 | Dejar pasajero 📤 |

### 4.1 · Crear el entorno y la Q-table de Taxi

Observa que el código es casi idéntico al de FrozenLake — solo cambia el nombre del entorno.

In [ ]:
env = gym.make("Taxi-v3", render_mode="rgb_array")

state_space  = env.observation_space.n
action_space = env.action_space.n
print(f"Estados posibles: {state_space}")
print(f"Acciones posibles: {action_space}")
print(f"Tamaño de la Q-table: {state_space} × {action_space} = {state_space * action_space} celdas")

# Reutilizamos la misma función de inicialización
Qtable_taxi = initialize_q_table(state_space, action_space)
print(f"\nQ-table creada: {Qtable_taxi.shape}  (todos en cero)")

### 4.2 · Hiperparámetros de Taxi

Los ajustes respecto a FrozenLake:
- **Más episodios** (25,000 vs 10,000): el espacio de estados es más grande, necesita más exploración
- **decay_rate más alto** (0.005 vs 0.0005): ε baja más rápido porque el espacio es manejable
- **eval_seed fijo**: garantiza que todos en el curso evalúan exactamente en los mismos estados

In [ ]:
# ── Hiperparámetros Taxi ──────────────────────────────────
n_training_episodes = 25_000
learning_rate       = 0.7
n_eval_episodes     = 100
env_id    = "Taxi-v3"
max_steps = 99
gamma     = 0.95

# Seed fijo para evaluación reproducible
# (no modificar — permite comparar resultados entre compañeros)
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,
             100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,161,
             131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,
             95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,
             145,152,97,106,55,31,85,38,112,102,168,123,97,21,83,158,26,80,
             63,5,81,32,11,28,148]

max_epsilon = 1.0
min_epsilon = 0.05
decay_rate  = 0.005   # más alto que FrozenLake

print("Hiperparámetros de Taxi configurados ✅")
print(f"Episodios: {n_training_episodes} | α: {learning_rate} | γ: {gamma} | decay: {decay_rate}")

### 4.3 · Entrenar en Taxi

Usamos **exactamente la misma función `train()`** que escribimos para FrozenLake.  
Esto tardará ~1-2 minutos.

In [ ]:
print("Entrenando agente en Taxi-v3...")
Qtable_taxi = train(
    n_training_episodes,
    min_epsilon,
    max_epsilon,
    decay_rate,
    env,
    max_steps,
    Qtable_taxi
)
print("✅ Entrenamiento completado")

### 4.4 · Evaluar y comparar

Para la certificación del curso de HF, la meta de Taxi es obtener una  
**recompensa media − desviación estándar ≥ 7.5**.

In [ ]:
print("Evaluando agente en Taxi-v3 (100 episodios con seed fijo)...")
mean_reward_taxi, std_reward_taxi = evaluate_agent(
    env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed
)
print(f"\nResultados Taxi-v3:")
print(f"  Recompensa media: {mean_reward_taxi:.2f}")
print(f"  Desviación estándar: ±{std_reward_taxi:.2f}")
print(f"  media - std: {mean_reward_taxi - std_reward_taxi:.2f}  (meta HF: ≥ 7.5)")

if mean_reward_taxi - std_reward_taxi >= 7.5:
    print("\n✅ ¡Meta de certificación alcanzada!")
else:
    print("\n🟡 Prueba con más episodios de entrenamiento o ajusta decay_rate.")
print()
print("─" * 40)
print("Comparativa final:")
print(f"  ❄️  FrozenLake: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  🚖 Taxi:       {mean_reward_taxi:.2f} ± {std_reward_taxi:.2f}")

### 4.5 · Ver al taxi en acción

In [ ]:
print("Grabando video del taxi...")
video_taxi = grabar_video_qtable('Taxi-v3', Qtable_taxi)
Video(video_taxi, embed=True, width=400)

---
## 5 · Experimenta: cambia parámetros y observa

La mejor forma de entender los hiperparámetros es cambiarlos y ver el efecto.  
Para ahorrar tiempo reducimos los episodios — el objetivo es ver **tendencias**, no resultados óptimos.

> ⏱️ Cada experimento tarda ~30 segundos con `n_training_episodes=3_000`.

### Experimento A · ¿Qué pasa con un learning_rate muy pequeño?

α = 0.7 aprende rápido pero puede ser inestable.  
α = 0.01 aprende muy despacio pero de forma más suave.  
¿Cuánto afecta con solo 3,000 episodios?

In [ ]:
env_exp = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='rgb_array')

# Experimento A1: learning_rate = 0.01 (muy lento)
Qtable_exp_a = initialize_q_table(env_exp.observation_space.n, env_exp.action_space.n)
learning_rate_original = learning_rate
learning_rate = 0.01  # muy pequeño
Qtable_exp_a = train(3_000, min_epsilon, max_epsilon, decay_rate, env_exp, max_steps, Qtable_exp_a)
mean_a, std_a = evaluate_agent(env_exp, max_steps, 100, Qtable_exp_a, [])

# Restaurar
learning_rate = learning_rate_original

print(f"α = 0.01 (lento):  {mean_a:.2f} ± {std_a:.2f}")
print(f"α = 0.70 (rápido): {mean_reward:.2f} ± {std_reward:.2f}  (del entrenamiento completo)")
print(f"\n→ ¿Cuál aprendió más en 3,000 episodios?")

### Experimento B · FrozenLake resbaladizo (`is_slippery=True`)

¿Recuerdas la versión resbaladiza del lago de las slides?  
El agente elige una dirección pero el hielo lo puede desviar a cualquier lado.  
¿Puede Q-Learning adaptarse?

In [ ]:
env_slip = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True, render_mode='rgb_array')
Qtable_slip = initialize_q_table(env_slip.observation_space.n, env_slip.action_space.n)

# Entrenar con los mismos hiperparámetros
Qtable_slip = train(10_000, min_epsilon, max_epsilon, decay_rate, env_slip, max_steps, Qtable_slip)
mean_slip, std_slip = evaluate_agent(env_slip, max_steps, 100, Qtable_slip, [])

print(f"FrozenLake NO resbaladizo: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"FrozenLake SÍ resbaladizo: {mean_slip:.2f} ± {std_slip:.2f}")
print()
print("→ ¿Por qué crees que el resultado es diferente?")
print("  Pista: con resbalo hay stochasticidad — el agente no siempre puede controlar",
        "su movimiento aunque elija la acción correcta.")

### Experimento C · ¿Qué pasa si epsilon no decae nunca?

Si `min_epsilon = max_epsilon = 1.0`, el agente siempre explora al azar — nunca explota.  
¿Puede aprender algo así?

In [ ]:
env_exp_c = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='rgb_array')
Qtable_exp_c = initialize_q_table(env_exp_c.observation_space.n, env_exp_c.action_space.n)

# Sin decaimiento: siempre ε = 1.0 (siempre aleatorio)
Qtable_exp_c = train(10_000, 1.0, 1.0, decay_rate, env_exp_c, max_steps, Qtable_exp_c)
mean_c, std_c = evaluate_agent(env_exp_c, max_steps, 100, Qtable_exp_c, [])

print(f"Con decaimiento ε (0.05→1.0): {mean_reward:.2f} ± {std_reward:.2f}")
print(f"Sin decaimiento  (ε fijo=1.0): {mean_c:.2f} ± {std_c:.2f}")
print()
print("→ La Q-table se actualiza correctamente, pero al evaluar se usa greedy_policy.")
print("  ¿Por qué el resultado es tan malo si la evaluación no usa ε?")

---

# 🔒 Secciones opcionales

> Las siguientes secciones requieren cuenta de Hugging Face y son completamente opcionales.  
> El objetivo principal del notebook ya está cumplido.

---

## [OPCIONAL] Publicar tu Q-table en el Hugging Face Hub

El Hub no solo acepta redes neuronales — también acepta Q-tables guardadas como archivo pickle.  
La función `push_to_hub` que definimos más abajo hace todo el proceso: evalúa, graba el video,  
genera la tarjeta del modelo y sube todo.

### Paso 1: Iniciar sesión

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # pega tu token con permiso Write

### Paso 2: Funciones de ayuda para el Hub
No necesitas entender este código ahora — es infraestructura para subir al Hub.

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save
from pathlib import Path
import datetime, json

def record_video(env, Qtable, out_directory, fps=1):
    """Graba un video de replay del agente."""
    images = []
    terminated = truncated = False
    state, _ = env.reset(seed=random.randint(0, 500))
    images.append(env.render())
    while not terminated and not truncated:
        action = np.argmax(Qtable[state][:])
        state, _, terminated, truncated, _ = env.step(action)
        images.append(env.render())
    imageio.mimsave(out_directory, [np.array(img) for img in images], fps=fps)

def push_to_hub(repo_id, model, env, video_fps=1, local_repo_path='hub'):
    """Evalúa, graba video y sube el modelo al Hub."""
    _, repo_name = repo_id.split('/')
    api = HfApi()
    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_local_path = Path(snapshot_download(repo_id=repo_id))
    if env.spec.kwargs.get('map_name'):
        model['map_name'] = env.spec.kwargs.get('map_name')
        if env.spec.kwargs.get('is_slippery', '') == False:
            model['slippery'] = False
    with open(repo_local_path / 'q-learning.pkl', 'wb') as f:
        pickle.dump(model, f)
    mean_reward_hub, std_reward_hub = evaluate_agent(
        env, model['max_steps'], model['n_eval_episodes'],
        model['qtable'], model['eval_seed']
    )
    with open(repo_local_path / 'results.json', 'w') as f:
        json.dump({'env_id': model['env_id'], 'mean_reward': mean_reward_hub,
                   'eval_datetime': datetime.datetime.now().isoformat()}, f)
    metadata = {'tags': [model['env_id'], 'q-learning', 'reinforcement-learning']}
    eval_meta = metadata_eval_result(
        model_pretty_name=repo_name, task_pretty_name='reinforcement-learning',
        task_id='reinforcement-learning', metrics_pretty_name='mean_reward',
        metrics_id='mean_reward',
        metrics_value=f'{mean_reward_hub:.2f} +/- {std_reward_hub:.2f}',
        dataset_pretty_name=model['env_id'], dataset_id=model['env_id'])
    metadata = {**metadata, **eval_meta}
    video_path = repo_local_path / 'replay.mp4'
    record_video(env, model['qtable'], video_path, video_fps)
    readme_path = repo_local_path / 'README.md'
    readme = f'# Q-Learning Agent — {model["env_id"]}\n'
    with readme_path.open('w', encoding='utf-8') as f: f.write(readme)
    metadata_save(readme_path, metadata)
    api.upload_folder(repo_id=repo_id, folder_path=repo_local_path, path_in_repo='.')
    print(f'Tu modelo está en el Hub: {repo_url}')

### Paso 3: Subir FrozenLake

In [ ]:
env_hub = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=False, render_mode='rgb_array')

model_fl = {
    'env_id': 'FrozenLake-v1',
    'max_steps': 99,
    'n_training_episodes': 10_000,
    'n_eval_episodes': 100,
    'eval_seed': [],
    'learning_rate': 0.7,
    'gamma': 0.95,
    'max_epsilon': 1.0,
    'min_epsilon': 0.05,
    'decay_rate': 0.0005,
    'qtable': Qtable_frozenlake
}

username  = ''   # ← pon tu usuario de HuggingFace
repo_name = 'q-FrozenLake-v1-4x4-noSlippery'
push_to_hub(f'{username}/{repo_name}', model_fl, env_hub)

### Paso 4: Subir Taxi

In [ ]:
env_hub_taxi = gym.make('Taxi-v3', render_mode='rgb_array')

model_taxi = {
    'env_id': 'Taxi-v3',
    'max_steps': 99,
    'n_training_episodes': 25_000,
    'n_eval_episodes': 100,
    'eval_seed': eval_seed,
    'learning_rate': 0.7,
    'gamma': 0.95,
    'max_epsilon': 1.0,
    'min_epsilon': 0.05,
    'decay_rate': 0.005,
    'qtable': Qtable_taxi
}

username  = ''   # ← pon tu usuario de HuggingFace
repo_name = 'q-Taxi-v3'
push_to_hub(f'{username}/{repo_name}', model_taxi, env_hub_taxi)

## [OPCIONAL] Cargar y evaluar modelos del Hub

Puedes descargar Q-tables entrenadas por la comunidad y evaluarlas.

In [ ]:
from urllib.error import HTTPError
from huggingface_hub import hf_hub_download

def load_from_hub(repo_id: str, filename: str):
    """Descarga una Q-table del Hugging Face Hub."""
    pkl_path = hf_hub_download(repo_id=repo_id, filename=filename)
    with open(pkl_path, 'rb') as f:
        return pickle.load(f)

# Descargar Q-table de Taxi del instructor del curso
model_hub = load_from_hub('ThomasSimonini/q-Taxi-v3', 'q-learning.pkl')
env_eval_hub = gym.make(model_hub['env_id'])
mean_hub, std_hub = evaluate_agent(
    env_eval_hub, model_hub['max_steps'],
    model_hub['n_eval_episodes'],
    model_hub['qtable'],
    model_hub['eval_seed']
)
print(f"Modelo de la comunidad (Taxi): {mean_hub:.2f} ± {std_hub:.2f}")
print(f"Tu modelo (Taxi):             {mean_reward_taxi:.2f} ± {std_reward_taxi:.2f}")

---

## 🎉 ¡Notebook completado!

Implementaste Q-Learning desde cero — sin ninguna librería de alto nivel.  
Lo que construiste hoy:

1. **`initialize_q_table()`** — crear la memoria del agente
2. **`greedy_policy()`** — explotar lo aprendido
3. **`epsilon_greedy_policy()`** — equilibrar exploración y explotación
4. **`train()`** — el bucle completo de Q-Learning con la ecuación de Bellman
5. **`evaluate_agent()`** — medir el desempeño de forma rigurosa

Y lo más importante: el mismo código resolvió tanto **FrozenLake** (16 estados)  
como **Taxi** (500 estados) sin cambiar una sola función.

### ¿Qué sigue?

- 🎮 **Módulo 3:** ¿Qué pasa cuando el espacio de estados es demasiado grande para una tabla?
  → Introducción a Deep Q-Learning (DQN) con redes neuronales
- 📓 **Notebook 3:** Demo de DQN jugando Space Invaders con RL-Baselines3-Zoo

---
*Aicraft · aicraft.mx · Jose Miguel Lara Rangel*